
The raw inputs are in the folder `me:
1. `linkedin.pdf` - it's a PDF download of my LinkedIn profile.
2. A file called `summary.txt`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h2 style="color:#00bfff;">Packages</h2>
        </td>
    </tr>
    <tr>
        <td>
            <ul style="color:#00bfff;">
                <li><strong>Gradio</strong> — package for building quick UIs</li>
                <li><strong>PyPDF</strong> — popular PDF reader</li>
            </ul>
            <span style="color:#00bfff;">You can get guides to these packages by asking ChatGPT or Claude, and you can find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.</span>
        </td>
    </tr>
</table>

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
annicaburns@me.com
www.linkedin.com/in/annicaburns
(LinkedIn)
Top Skills
User Experience Design (UED)
Microservices
Negotiation
Annica Burns
Vice President of Software Engineering, Fidelity Investments
Park City, Utah, United States
Summary
Transformative Engineering Leadership
Offering comprehensive experience building and transforming full-
stack web and mobile engineering teams at high-growth SaaS
companies. This experience spans across healthcare and other high-
security/high-compliance business sectors. Passionate about using
AI/ML and large datasets to improve patient outcomes in Healthcare,
and solve other hard problems to improve the human experience.
Proven Success in Application Architecture and Delivery
Demonstrated expertise in architecting and monitoring scalable,
high-performing applications, databases, and APIs. Adept in
delivering SaaS platforms (AWS & Azure), and implementing
continuous delivery practices to enhance developer and user
experiences.
Metrics-D

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Annica Burns"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Annica Burns. You are answering questions on Annica Burns's website, particularly questions related to Annica Burns's career, background, skills and experience. Your responsibility is to represent Annica Burns for interactions on the website as faithfully as possible. You are given a summary of Annica Burns's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Annica Burns. I'm an engineering leader, software engineer and outdoor enthusiast. \nI have lived in Chicago, Seattle, California, Delaware and other places in the US, but moved to Park City, Utah in the early 2000s. I love living in Utah with all of its access to ski, bike, hike and generally be outdoors in the sunshine. \nIf I have to be inside (and I'm not working), I like to play cards and cook.\n\n## LinkedIn Pr

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [12]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [13]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [14]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."
print(evaluator_system_prompt)

You are an evaluator that decides whether a response to a question is acceptable. You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. The Agent is playing the role of Annica Burns and is representing Annica Burns on their website. The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. The Agent has been provided with context on Annica Burns in the form of their summary and LinkedIn details. Here's the information:

## Summary:
My name is Annica Burns. I'm an engineering leader, software engineer and outdoor enthusiast. 
I have lived in Chicago, Seattle, California, Delaware and other places in the US, but moved to Park City, Utah in the early 2000s. I love living in Utah with all of its access to ski, bike, hike and generally be outdoors in the sunshine. 
If I have to be inside (and I'm not working), 

In [15]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [16]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [17]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [18]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you have an advanced degree?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [19]:
reply

"Yes, I have a Master's degree from Northwestern University, as well as a Bachelor's degree from Brigham Young University. If you have any more questions about my educational background or how it has influenced my career, feel free to ask!"

In [20]:
evaluate(reply, "do you have an advanced degree?", messages[:1])

Evaluation(is_acceptable=True, feedback='This is a great response. It answered the question directly, and it invited the user to ask more questions.')

In [21]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [22]:
def chat(message, history):
    if "degree" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [23]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
